# 自行车共享数据分析

## 问题定义

- 哪个季节的自行车租赁数量最高？
- 每个月和每年的自行车租赁模式如何？
- 特定日期的自行车租赁模式如何？


## 准备库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import calendar

## 数据处理

### 数据收集

In [ ]:
# 加载每日数据
day_df = pd.read_csv("day.csv")
day_df.head()

In [ ]:
# 加载每小时数据
hour_df = pd.read_csv("hour.csv")
hour_df.head()

### 数据评估

#### 评估 day_df

In [ ]:
# 查看 day_df 数据信息
day_df.info()

In [ ]:
# 检查 day_df 缺失值
day_df.isna().sum()

In [ ]:
# 查看 day_df 数据重复数量
print("重复数量: ", day_df.duplicated().sum())

In [ ]:
# 查看 day_df 列的统计信息
day_df.describe()

#### 评估 hour_df

In [ ]:
# 查看 hour_df 数据信息
hour_df.info()

In [ ]:
#  检查 hour_df 缺失值
hour_df.isna().sum()

In [ ]:
# 查看 hour_df 数据重复数量
print("重复数量: ", hour_df.duplicated().sum())


In [ ]:
# 查看 hour_df 列的统计信息
hour_df.describe()


### 3. 清洗数据

#### 修改数据类型

In [ ]:
# 将 "dteday" 列的数据类型改为 datetime
day_df["dteday"] = pd.to_datetime(day_df["dteday"])
hour_df["dteday"] = pd.to_datetime(hour_df["dteday"])

In [ ]:
# 查看 day_df 数据信息
day_df.info()

In [ ]:
# 查看 hour_df 数据信息
hour_df.info()

#### 修改列名

In [ ]:
# 将列 'dteday', 'yr', 'mnth', 'hr', 'temp', 'hum', 'cnt' 重命名
def rename_columns(dataframe, column_mapping):
    for old_col, new_col in column_mapping.items():
        dataframe.rename(columns={old_col: new_col}, inplace=True)

column_day = {'dteday': 'date', 'yr': 'year', 'mnth': 'month', 'temp': 'temperature', 'hum': 'humidity', 'cnt': 'total'}
column_hour = {'dteday': 'date', 'yr': 'year', 'mnth': 'month', 'hr': 'hour', 'temp': 'temperature', 'hum': 'humidity', 'cnt': 'total'}

rename_columns(day_df, column_day)
rename_columns(hour_df, column_hour)

#### 修改季节分类的名称

In [ ]:
season_mapping = {1: '春季', 2: '夏季', 3: '秋季', 4: '冬季'}
day_df['season'] = day_df['season'].replace(season_mapping)
hour_df['season'] = hour_df['season'].replace(season_mapping)

#### 修改月份分类的名称

In [ ]:
day_df['month'] = day_df['month'].apply(lambda x: calendar.month_name[x])
hour_df['month'] = hour_df['month'].apply(lambda x: calendar.month_name[x])

#### 修改年份类别名称

In [ ]:
year_mapping = {0: '2011年', 1: '2012年'}
day_df['year'] = day_df['year'].replace(year_mapping)
hour_df['year'] = hour_df['year'].replace(year_mapping)

#### 修改星期分类的名称

In [ ]:
weekday_mapping = {0: '星期日', 1: '星期一', 2: '星期二', 3: '星期三', 4: '星期四', 5: '星期五', 6: '星期六'}
day_df['weekday'] = day_df['weekday'].replace(weekday_mapping)
hour_df['weekday'] = hour_df['weekday'].replace(weekday_mapping)

In [ ]:
# 查看day_df的前几行
day_df.head()

In [ ]:
# 查看hour_df的前几行数据
hour_df.head()

## 探索性数据分析（EDA）

#### 根据季节的自行车使用统计

In [ ]:
day_df.groupby(by="season").agg({
    "casual": ["max", "min", "mean", "sum"],
    "registered": ["max", "min", "mean", "sum"],
    "total": ["max", "min", "mean", "sum"]
})

#### 根据月份和年份的自行车使用统计

In [ ]:
month_order = {month: i for i, month in enumerate(calendar.month_name[1:], start=1)}
day_df['month'] = pd.Categorical(day_df['month'], categories=sorted(month_order, key=month_order.get))

day_df.groupby(by=["year", "month"], observed=False).agg({
    "casual": ["max", "min", "mean", "sum"],
    "registered": ["max", "min", "mean", "sum"],
    "total": ["max", "min", "mean", "sum"]
})

#### 每个节假日的自行车使用量

In [ ]:
hour_df[(hour_df["holiday"] == 1) | (hour_df["workingday"] == 0)].groupby(by=["weekday"], observed=False).agg({"total": ["sum"]})

#### 每个工作日的自行车使用量

In [ ]:
hour_df[(hour_df["holiday"] == 0) & (hour_df["workingday"] == 1)].groupby(by=["weekday"], observed=False).agg({"total": ["sum"]})

## 数据可视化

### 问题1：在哪个季节自行车使用量最高？

In [ ]:
plot_season = day_df.groupby('season')[['registered', 'casual']].sum().reset_index()

plt.figure(figsize=(10, 6))

bar_width = 0.35
bar_positions1 = range(len(plot_season['season']))
bar_positions2 = [pos + bar_width for pos in bar_positions1]

plt.bar(bar_positions1, plot_season['registered'], width=bar_width, label='注册用户', color='tab:blue')
plt.bar(bar_positions2, plot_season['casual'], width=bar_width, label='非注册用户', color='tab:orange')

plt.xlabel('季节')
plt.ylabel('租赁总数')
plt.title('自行车租赁总数按季节分布', fontsize = 17)
plt.xticks([pos + bar_width/2 for pos in bar_positions1], plot_season['season'])
plt.legend()
plt.show()

### 问题2：在每个月和每年的自行车租赁模式如何变化？

In [ ]:
plt.figure(figsize=(10, 6))

plot_ym = day_df.groupby(by=["month", "year"], observed=False).agg({
    "total": "sum"
}).reset_index()

sns.lineplot(
    data=plot_ym,
    x="month",
    y="total",
    hue="year",
    style="year",  
    markers=True,
    markersize=8,  
    dashes=False,  
)

plt.grid(False)
plt.title("自行车租赁数量按月份和年份分布", fontsize = 17)
plt.xlabel(None)
plt.ylabel(None)
plt.legend(title="年份", loc="upper right")
plt.tight_layout()

plt.show()


### 问题3：在特定日期的自行车租赁模式如何变化？

In [ ]:
plt.figure(figsize=(30, 10))

mask1 = ((hour_df['workingday'] == 0) | (hour_df['holiday'] == 1))
df1 = hour_df[mask1]
mask2 = ((hour_df['workingday'] == 1) & (hour_df['holiday'] == 0))
df2 = hour_df[mask2]

plot_weekday = sns.FacetGrid(hour_df, col='weekday', hue='workingday', col_wrap=2, height=5, sharex=False)
plot_weekday.map(sns.lineplot, "hour", "total")
plot_weekday.fig.suptitle('自行车租赁数量按工作日和节假日分布', fontsize=20, y=1.02, fontsize=20)
plot_weekday.set_axis_labels('小时', '租赁总数')
plot_weekday.add_legend()

for ax in plot_weekday.axes.flat:
    ax.set(xlabel="小时")
plot_weekday.set_titles(size=10)

plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.show()

### 结论

#### 问题1:在什么季节自行车租赁数量最高?
秋季的自行车租赁数量是其他季节中最高的，即在秋季和注册用户中，自行车租租赁数量超过100万。另一方面，春季自行车租赁数量最低，只有471,348。另一方面，在每个季节，登记用户主导自行车租赁的数量。


#### 问题2:租自行车的模式是怎样的?
2012年自行车租金比2011年高。此外，自行车租赁模式在年初至第三季度有所增加，然后在年底有所下降。

#### 问题3:租自行车的时间有什么规律? 
在工作日，自行车租赁在某些时间是很高的，比如5点到8点，15点到17点。这表明工作时间的流逝与自行车租赁数量的增加有关。此外，在周末，自行车租赁的数量往往在7点到3点之间增加。而在工作日，自行车租赁的数量则异常波动。